In [0]:
# Cell 1: Imports and load Chicago bronze table
from pyspark.sql import functions as F
from pyspark.sql.types import *

chicago_bronze = spark.table("food_inspections.bronze.chicago_inspections")
print(f"Chicago Bronze: {chicago_bronze.count()} rows, {len(chicago_bronze.columns)} columns")
chicago_bronze.printSchema()

Chicago Bronze: 308357 rows, 20 columns
root
 |-- inspection_id: string (nullable = true)
 |-- dba_name: string (nullable = true)
 |-- aka_name: string (nullable = true)
 |-- license_num: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- risk: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- inspection_date: string (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- results: string (nullable = true)
 |-- violations: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- location: string (nullable = true)
 |-- source_city: string (nullable = true)
 |-- load_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
# Cell 2: Chicago first-pass filters -- drop rows that fail basic validation
chicago_filtered = chicago_bronze.filter(
    # Rule 1: restaurant name cannot be null
    (F.col("dba_name").isNotNull()) & (F.trim(F.col("dba_name")) != "") &
    # Rule 1: inspection date cannot be null
    (F.col("inspection_date").isNotNull()) & (F.trim(F.col("inspection_date")) != "") &
    # Rule 1: inspection type cannot be null
    (F.col("inspection_type").isNotNull()) & (F.trim(F.col("inspection_type")) != "") &
    # Rule 2: zip cannot be null and must be valid 5-digit
    (F.col("zip").isNotNull()) & (F.col("zip").rlike("^\\d{5}$")) &
    # Rule 4: Chicago results cannot be null
    (F.col("results").isNotNull()) & (F.trim(F.col("results")) != "") &
    # Rule 5: must have at least 1 violation (non-null, non-empty)
    (F.col("violations").isNotNull()) & (F.trim(F.col("violations")) != "")
)

before_count = chicago_bronze.count()
after_count = chicago_filtered.count()
dropped = before_count - after_count
print(f"Before: {before_count}")
print(f"After:  {after_count}")
print(f"Dropped: {dropped} ({dropped/before_count*100:.1f}%)")

Before: 308357
After:  221873
Dropped: 86484 (28.0%)


In [0]:
# Cell 3: Chicago text cleanup, type casting, inspection_type standardization
from pyspark.sql import functions as F

# Inspection type canonical mapping (upper case applied first, then these fix typos/variants)
inspection_type_mapping = {
    "CANVAS": "CANVASS",
    "CANVASS RE-INSPECTION": "CANVASS RE-INSPECTION",
    "CANVASS REINSPECTION": "CANVASS RE-INSPECTION",
    "LICENSE REINSPECTION": "LICENSE RE-INSPECTION",
    "LICENSE RE-INSPECTON": "LICENSE RE-INSPECTION",
    "COMPLAINT RE-INSPECTION": "COMPLAINT RE-INSPECTION",
    "COMPLAINT REINSPECTION": "COMPLAINT RE-INSPECTION",
    "COMPLIANT": "COMPLAINT",
    "COMPLIANT RE-INSPECTION": "COMPLAINT RE-INSPECTION",
    "OUT OFBUSINESS": "OUT OF BUSINESS",
    "OUT OF BUSINES": "OUT OF BUSINESS",
    "OUTOFBUSINESS": "OUT OF BUSINESS",
    "NON-INSPECTION": "NON-INSPECTION",
    "NONINSPECTION": "NON-INSPECTION",
    "SHORT FORM COMPLAINT": "SHORT FORM COMPLAINT",
    "SHORTFORM COMPLAINT": "SHORT FORM COMPLAINT",
    "NO ENTRY": "NO ENTRY",
    "NOENTRY": "NO ENTRY",
    "SUSPECTED FOOD POISONING": "SUSPECTED FOOD POISONING",
    "TASK": "TASK",
    "CONSULTATION": "CONSULTATION",
    "TAG REMOVAL": "TAG REMOVAL",
    "RECENT INSPECTION": "RECENT INSPECTION",
}

# Build a chained when/otherwise for the mapping
type_expr = F.upper(F.trim(F.col("inspection_type")))
for bad, good in inspection_type_mapping.items():
    type_expr = F.when(F.upper(F.trim(F.col("inspection_type"))) == bad, F.lit(good)).otherwise(type_expr)

# Violation score derivation (Rule 8)
score_expr = (
    F.when(F.upper(F.trim(F.col("results"))) == "PASS", 90)
     .when(F.upper(F.trim(F.col("results"))) == "PASS W/ CONDITIONS", 80)
     .when(F.upper(F.trim(F.col("results"))) == "FAIL", 70)
     .when(F.upper(F.trim(F.col("results"))) == "NO ENTRY", 0)
     .otherwise(F.lit(None).cast("double"))
)

# Pass/fail flag
pass_fail_expr = (
    F.when(F.upper(F.trim(F.col("results"))).isin("PASS", "PASS W/ CONDITIONS"), "PASS")
     .when(F.upper(F.trim(F.col("results"))) == "FAIL", "FAIL")
     .otherwise("OTHER")
)

chicago_cleaned = chicago_filtered.select(
    F.trim(F.col("inspection_id")).alias("inspection_id"),
    F.upper(F.trim(F.col("dba_name"))).alias("dba_name"),
    F.upper(F.trim(F.col("aka_name"))).alias("aka_name"),
    F.trim(F.col("license_num")).alias("license_num"),
    F.upper(F.trim(F.col("facility_type"))).alias("facility_type"),
    F.trim(F.col("risk")).alias("risk"),
    F.upper(F.trim(F.col("address"))).alias("address"),
    F.upper(F.trim(F.col("city"))).alias("city"),
    F.upper(F.trim(F.col("state"))).alias("state"),
    F.trim(F.col("zip")).alias("zip"),
    F.to_date(F.col("inspection_date"), "MM/dd/yyyy").alias("inspection_date"),
    type_expr.alias("inspection_type"),
    F.upper(F.trim(F.col("results"))).alias("results"),
    F.col("violations"),
    F.col("latitude").cast("double").alias("latitude"),
    F.col("longitude").cast("double").alias("longitude"),
    score_expr.alias("violation_score"),
    pass_fail_expr.alias("pass_fail_flag"),
    F.trim(F.col("license_num")).alias("restaurant_nk"),
    F.trim(F.col("inspection_id")).alias("inspection_id_nk"),
    F.col("source_city"),
    F.current_timestamp().alias("load_ts"),
)

print(f"Chicago cleaned: {chicago_cleaned.count()} rows")

# Verify inspection_type standardization
print("\nInspection types after standardization:")
chicago_cleaned.groupBy("inspection_type").count().orderBy(F.desc("count")).show(30, truncate=False)

# Verify violation_score derivation
print("\nViolation score by result:")
chicago_cleaned.groupBy("results", "violation_score", "pass_fail_flag").count().orderBy(F.desc("count")).show(15, truncate=False)

# Check for null inspection_dates (bad date parsing)
null_dates = chicago_cleaned.filter(F.col("inspection_date").isNull()).count()
print(f"\nNull inspection_dates after parsing: {null_dates}")

Chicago cleaned: 221873 rows

Inspection types after standardization:
+--------------------------------------+------+
|inspection_type                       |count |
+--------------------------------------+------+
|CANVASS                               |121697|
|COMPLAINT                             |26525 |
|CANVASS RE-INSPECTION                 |24729 |
|LICENSE                               |24484 |
|COMPLAINT RE-INSPECTION               |8696  |
|SHORT FORM COMPLAINT                  |6546  |
|LICENSE RE-INSPECTION                 |5957  |
|SUSPECTED FOOD POISONING              |958   |
|TAG REMOVAL                           |496   |
|CONSULTATION                          |401   |
|RECENT INSPECTION                     |390   |
|LICENSE-TASK FORCE                    |291   |
|SUSPECTED FOOD POISONING RE-INSPECTION|166   |
|COMPLAINT-FIRE                        |122   |
|NON-INSPECTION                        |119   |
|TASK FORCE LIQUOR 1475                |95    |
|SHORT FORM FIRE-C

In [0]:
# Cell 4: Parse Chicago violations blob into individual violation rows
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType, StructField, StringType

# Step 1: Split violations on ' | ' (pipe with surrounding spaces)
chicago_exploded = chicago_cleaned.withColumn(
    "violation_raw", F.explode(F.split(F.col("violations"), "\\s*\\|\\s*"))
).filter(
    # Drop empty segments from trailing pipes
    (F.col("violation_raw").isNotNull()) & (F.trim(F.col("violation_raw")) != "")
)

# Step 2: Extract violation_code, violation_description, violation_text (comments)
# Pattern: "CODE. DESCRIPTION - Comments: TEXT"
# Some segments may not have "- Comments:" at all
chicago_violations = chicago_exploded.select(
    F.col("inspection_id_nk"),
    F.col("inspection_id"),
    F.col("violation_raw"),
    # Extract code: leading digits before the first period
    F.regexp_extract(F.col("violation_raw"), r"^(\d+)\.", 1).alias("violation_code"),
    # Extract description: text between "CODE. " and " - Comments:" (or end of string)
    F.regexp_extract(
        F.col("violation_raw"),
        r"^\d+\.\s*(.*?)(?:\s*-\s*Comments\s*:.*$|$)",
        1
    ).alias("violation_description"),
    # Extract comments text: everything after "- Comments:"
    F.regexp_extract(
        F.col("violation_raw"),
        r"-\s*Comments\s*:\s*(.*)",
        1
    ).alias("violation_text"),
    F.col("source_city"),
    F.current_timestamp().alias("load_ts"),
)

# Add violation_order (sequence number within each inspection)
from pyspark.sql.window import Window

w = Window.partitionBy("inspection_id_nk").orderBy("violation_code")
chicago_violations = chicago_violations.withColumn(
    "violation_order", F.row_number().over(w)
)

print(f"Chicago violations (exploded): {chicago_violations.count()} rows")
print(f"From {chicago_violations.select('inspection_id_nk').distinct().count()} inspections")

# Check parsing quality
print("\nSample parsed violations:")
chicago_violations.select(
    "inspection_id_nk", "violation_code", "violation_description", "violation_text"
).show(5, truncate=80)

# Check for empty violation_codes (parsing failures)
empty_codes = chicago_violations.filter(
    (F.col("violation_code").isNull()) | (F.col("violation_code") == "")
).count()
print(f"\nEmpty violation codes (parse failures): {empty_codes}")

# Violations per inspection distribution
print("\nViolations per inspection:")
chicago_violations.groupBy("inspection_id_nk").count().select(
    F.min("count").alias("min"),
    F.max("count").alias("max"),
    F.round(F.avg("count"), 1).alias("avg"),
    F.expr("percentile_approx(count, 0.5)").alias("median"),
).show()

Chicago violations (exploded): 1001962 rows
From 221873 inspections

Sample parsed violations:
+----------------+--------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|inspection_id_nk|violation_code|                                                           violation_description|                                                                  violation_text|
+----------------+--------------+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|         2633568|             3|MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, RESPONSIBILITI...|OBSERVED NO SIGNED EMPLOYEES HEALTH POLICIES MUST PROVIDE AND MAINTAIN.(PRIOR...|
|         2633568|            10|                     ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE|OBSERVE

In [0]:
# Cell 5 (fixed): Flag critical violations + apply Rule 7 (pure PASS only)
from pyspark.sql import functions as F

# Critical count per inspection (already computed from chicago_violations)
critical_per_inspection = chicago_violations.groupBy("inspection_id_nk").agg(
    F.sum(F.when(F.col("is_critical_flag") == "Y", 1).otherwise(0)).alias("critical_count")
)

# Join back to inspection-level
chicago_insp_with_critical = chicago_cleaned.join(
    critical_per_inspection,
    chicago_cleaned["inspection_id_nk"] == critical_per_inspection["inspection_id_nk"],
    "inner"
).drop(critical_per_inspection["inspection_id_nk"])

# Rule 7: only pure PASS (not PASS W/ CONDITIONS) with critical violations
rule7_violations = chicago_insp_with_critical.filter(
    (F.col("results") == "PASS") & (F.col("critical_count") > 0)
).count()
print(f"Rule 7 violations (pure PASS with critical): {rule7_violations}")

# Apply Rule 7
chicago_inspections_silver = chicago_insp_with_critical.filter(
    ~((F.col("results") == "PASS") & (F.col("critical_count") > 0))
).drop("critical_count")

print(f"Before Rule 7: {chicago_insp_with_critical.count()}")
print(f"After Rule 7:  {chicago_inspections_silver.count()}")
print(f"Dropped by Rule 7: {rule7_violations}")

# Sanity check: result distribution after Rule 7
print("\nResult distribution after all filters:")
chicago_inspections_silver.groupBy("results", "pass_fail_flag").count().orderBy(F.desc("count")).show()

Rule 7 violations (pure PASS with critical): 427
Before Rule 7: 221873
After Rule 7:  221446
Dropped by Rule 7: 427

Result distribution after all filters:
+------------------+--------------+------+
|           results|pass_fail_flag| count|
+------------------+--------------+------+
|              PASS|          PASS|119710|
|              FAIL|          FAIL| 55836|
|PASS W/ CONDITIONS|          PASS| 45010|
|          NO ENTRY|         OTHER|   765|
|         NOT READY|         OTHER|    82|
|   OUT OF BUSINESS|         OTHER|    43|
+------------------+--------------+------+



In [0]:
# Cell 6: Finalize and write Chicago Silver tables
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Get the surviving inspection IDs
surviving_ids = chicago_inspections_silver.select("inspection_id_nk").distinct()

# Filter violations to only surviving inspections
chicago_violations_silver = chicago_violations.join(
    surviving_ids,
    on="inspection_id_nk",
    how="inner"
)

# Recompute violation_order after filtering (in case dropped inspections affected anything)
w = Window.partitionBy("inspection_id_nk").orderBy("violation_code")
chicago_violations_silver = chicago_violations_silver.drop("violation_order").withColumn(
    "violation_order", F.row_number().over(w)
)

# Count total violations per inspection
violation_counts = chicago_violations_silver.groupBy("inspection_id_nk").agg(
    F.count("*").alias("total_violations")
)

# Add total_violations to inspection table
chicago_inspections_silver = chicago_inspections_silver.join(
    violation_counts,
    on="inspection_id_nk",
    how="left"
)

# Final column selection for inspection table (drop raw violations blob)
chicago_inspections_final = chicago_inspections_silver.select(
    "inspection_id_nk",
    "inspection_id",
    "restaurant_nk",
    "dba_name",
    "aka_name",
    "license_num",
    "facility_type",
    "risk",
    "address",
    "city",
    "state",
    "zip",
    "inspection_date",
    "inspection_type",
    "results",
    "violation_score",
    "pass_fail_flag",
    "total_violations",
    "latitude",
    "longitude",
    "source_city",
    "load_ts",
)

# Final column selection for violations table (drop violation_raw)
chicago_violations_final = chicago_violations_silver.select(
    "inspection_id_nk",
    "violation_order",
    "violation_code",
    "violation_description",
    "violation_text",
    "is_critical_flag",
    "source_city",
    "load_ts",
)

# Write inspection table
chicago_inspections_final.write.mode("overwrite").saveAsTable(
    "food_inspections.silver.chicago_inspections"
)

# Write violations table
chicago_violations_final.write.mode("overwrite").saveAsTable(
    "food_inspections.silver.chicago_violations"
)

# Verify
ci = spark.table("food_inspections.silver.chicago_inspections")
cv = spark.table("food_inspections.silver.chicago_violations")
print(f"Chicago Silver inspections: {ci.count()} rows, {len(ci.columns)} cols")
print(f"Chicago Silver violations:  {cv.count()} rows, {len(cv.columns)} cols")

# Every inspection should have at least 1 violation
zero_viol = ci.filter(F.col("total_violations") == 0).count()
print(f"Inspections with 0 violations: {zero_viol}")

# Every violation should map to a surviving inspection
orphan_viols = cv.join(ci.select("inspection_id_nk"), on="inspection_id_nk", how="left_anti").count()
print(f"Orphan violations: {orphan_viols}")

print("\nChicago Silver inspections schema:")
ci.printSchema()

Chicago Silver inspections: 221446 rows, 22 cols
Chicago Silver violations:  1000213 rows, 8 cols
Inspections with 0 violations: 0
Orphan violations: 0

Chicago Silver inspections schema:
root
 |-- inspection_id_nk: string (nullable = true)
 |-- inspection_id: string (nullable = true)
 |-- restaurant_nk: string (nullable = true)
 |-- dba_name: string (nullable = true)
 |-- aka_name: string (nullable = true)
 |-- license_num: string (nullable = true)
 |-- facility_type: string (nullable = true)
 |-- risk: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- inspection_date: date (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- results: string (nullable = true)
 |-- violation_score: double (nullable = true)
 |-- pass_fail_flag: string (nullable = true)
 |-- total_violations: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- lon

# Dallas

In [0]:
# Cell 7: Read Dallas bronze table
from pyspark.sql import functions as F

dallas_bronze = spark.table("food_inspections.bronze.dallas_inspections")
print(f"Dallas Bronze: {dallas_bronze.count()} rows, {len(dallas_bronze.columns)} columns")

# Show just the core columns (not all 117)
core_cols = ["restaurant_name", "inspection_type", "inspection_date", "inspection_score",
             "street_address", "zip_code", "lat_long_location", "source_city", "load_ts"]
dallas_bronze.select(core_cols).show(5, truncate=60)

Dallas Bronze: 78984 rows, 117 columns
+-------------------------------------------+---------------+---------------+----------------+-------------------------+--------+-----------------------+-----------+--------------------------+
|                            restaurant_name|inspection_type|inspection_date|inspection_score|           street_address|zip_code|      lat_long_location|source_city|                   load_ts|
+-------------------------------------------+---------------+---------------+----------------+-------------------------+--------+-----------------------+-----------+--------------------------+
|TIENDA Y RESTAURANTE LA CAMPIñA SALvadorena|        Routine|     10/03/2016|              66|      10818 DENNIS RD 102|   75229|(32.895847, -96.881391)|     DALLAS|2026-04-08 17:28:18.228109|
|                      TORTILLERIA LA ESPIGA|        Routine|     10/03/2016|              82|1328 N JIM MILLER RD #104|   75217| (32.73556, -96.700079)|     DALLAS|2026-04-08 17:28:18.2281

In [0]:
# Cell 8: Dallas first-pass filters -- drop rows that fail basic validation
from pyspark.sql import functions as F

# Build a count of non-null, non-empty violation descriptions across all 25 slots
violation_count_expr = sum(
    F.when(
        (F.col(f"violation_description_{i}").isNotNull()) &
        (F.trim(F.col(f"violation_description_{i}")) != ""),
        1
    ).otherwise(0)
    for i in range(1, 26)
)

dallas_with_viol_count = dallas_bronze.withColumn("raw_violation_count", violation_count_expr)

dallas_filtered = dallas_with_viol_count.filter(
    # Rule 1: restaurant name cannot be null
    (F.col("restaurant_name").isNotNull()) & (F.trim(F.col("restaurant_name")) != "") &
    # Rule 1: inspection date cannot be null
    (F.col("inspection_date").isNotNull()) & (F.trim(F.col("inspection_date")) != "") &
    # Rule 1: inspection type cannot be null
    (F.col("inspection_type").isNotNull()) & (F.trim(F.col("inspection_type")) != "") &
    # Rule 2: zip cannot be null and must be valid 5-digit
    (F.col("zip_code").isNotNull()) & (F.col("zip_code").rlike("^\\d{5}$")) &
    # Rule 3: Dallas score cannot be more than 100
    (F.col("inspection_score").cast("int") <= 100) &
    # Negative scores (data entry errors)
    (F.col("inspection_score").cast("int") >= 0) &
    # Rule 5: must have at least 1 violation
    (F.col("raw_violation_count") > 0)
)

before_count = dallas_bronze.count()
after_count = dallas_filtered.count()
dropped = before_count - after_count
print(f"Before: {before_count}")
print(f"After:  {after_count}")
print(f"Dropped: {dropped} ({dropped/before_count*100:.1f}%)")

# Breakdown of what got dropped
print("\nDrop breakdown:")
print(f"  Null restaurant_name: {dallas_bronze.filter(F.col('restaurant_name').isNull() | (F.trim(F.col('restaurant_name')) == '')).count()}")
print(f"  Invalid zip: {dallas_bronze.filter(~F.col('zip_code').rlike('^\\\\d{{5}}$')).count()}")
print(f"  Negative score: {dallas_bronze.filter(F.col('inspection_score').cast('int') < 0).count()}")
print(f"  Score > 100: {dallas_bronze.filter(F.col('inspection_score').cast('int') > 100).count()}")
print(f"  Zero violations: {dallas_with_viol_count.filter(F.col('raw_violation_count') == 0).count()}")

Before: 78984
After:  72159
Dropped: 6825 (8.6%)

Drop breakdown:
  Null restaurant_name: 11


---------------------------------------------------------------------------
SparkRuntimeException                     Traceback (most recent call last)
File <command-6049220915290566>, line 43
     41 print("\nDrop breakdown:")
     42 print(f"  Null restaurant_name: {dallas_bronze.filter(F.col('restaurant_name').isNull() | (F.trim(F.col('restaurant_name')) == '')).count()}")
---> 43 print(f"  Invalid zip: {dallas_bronze.filter(~F.col('zip_code').rlike('^\\\\d{{5}}$')).count()}")
     44 print(f"  Negative score: {dallas_bronze.filter(F.col('inspection_score').cast('int') < 0).count()}")
     45 print(f"  Score > 100: {dallas_bronze.filter(F.col('inspection_score').cast('int') > 100).count()}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:318, in DataFrame.count(self)
    315 def count(self) -> int:
    316     table, _ = self.agg(
    317         F._invoke_function("count", F.lit(1))
--> 318     )._to_table()  # type: ignore[operator]
    319 

In [0]:
# Cell 9: Dallas text cleanup, type casting, derived columns, key generation
from pyspark.sql import functions as F

# Clean and cast core columns
dallas_cleaned = dallas_filtered.select(
    F.upper(F.trim(F.col("restaurant_name"))).alias("restaurant_name"),
    F.col("street_number"),
    F.col("street_name"),
    F.col("street_direction"),
    F.col("street_type"),
    F.col("street_unit"),
    F.upper(F.trim(F.col("street_address"))).alias("street_address"),
    F.trim(F.col("zip_code")).alias("zip_code"),
    F.upper(F.trim(F.col("inspection_type"))).alias("inspection_type"),
    F.to_date(F.col("inspection_date"), "MM/dd/yyyy").alias("inspection_date"),
    F.col("inspection_score").cast("int").alias("inspection_score"),
    F.when(F.col("inspection_score").cast("int") >= 70, "PASS")
     .otherwise("FAIL").alias("results"),
    F.when(F.col("inspection_score").cast("int") >= 70, "PASS")
     .otherwise("FAIL").alias("pass_fail_flag"),
    F.col("inspection_score").cast("double").alias("violation_score"),
    F.regexp_extract(F.col("lat_long_location"), r"\(\s*(-?[\d.]+)\s*,", 1)
     .cast("double").alias("latitude"),
    F.regexp_extract(F.col("lat_long_location"), r",\s*(-?[\d.]+)\s*\)", 1)
     .cast("double").alias("longitude"),
    F.col("raw_violation_count"),
    *[F.col(f"violation_description_{i}") for i in range(1, 26)],
    *[F.col(f"violation_points_{i}") for i in range(1, 26)],
    *[F.col(f"violation_detail_{i}") for i in range(1, 26)],
    *[F.col(f"violation_memo_{i}") for i in range(1, 26)],
    F.col("source_city"),
    F.current_timestamp().alias("load_ts"),
)

dallas_cleaned = dallas_cleaned.withColumn(
    "restaurant_nk",
    F.md5(F.concat(
        F.col("restaurant_name"),
        F.lit("||"),
        F.col("street_address"),
        F.lit("||"),
        F.col("zip_code"),
    ))
)

dallas_cleaned = dallas_cleaned.withColumn(
    "inspection_id_nk",
    F.md5(F.concat(
        F.col("restaurant_nk"),
        F.lit("||"),
        F.col("inspection_date").cast("string"),
        F.lit("||"),
        F.col("inspection_score").cast("string"),
    ))
)

print(f"Dallas cleaned: {dallas_cleaned.count()} rows")

null_dates = dallas_cleaned.filter(F.col("inspection_date").isNull()).count()
print(f"Null inspection_dates: {null_dates}")

null_coords = dallas_cleaned.filter(F.col("latitude").isNull()).count()
print(f"Null coordinates: {null_coords}")

print("\nResult distribution:")
dallas_cleaned.groupBy("results", "pass_fail_flag").count().orderBy(F.desc("count")).show()

total = dallas_cleaned.count()
unique_ids = dallas_cleaned.select("inspection_id_nk").distinct().count()
print(f"Total rows: {total}, Unique inspection_id_nk: {unique_ids}, Dupes: {total - unique_ids}")

Dallas cleaned: 72159 rows
Null inspection_dates: 0
Null coordinates: 7626

Result distribution:
+-------+--------------+-----+
|results|pass_fail_flag|count|
+-------+--------------+-----+
|   PASS|          PASS|71654|
|   FAIL|          FAIL|  505|
+-------+--------------+-----+

Total rows: 72159, Unique inspection_id_nk: 72073, Dupes: 86


In [0]:
# Cell 10: Unpivot Dallas violations from 25 wide columns to tall rows
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql import DataFrame

# Build one dataframe per violation slot, then union all
violation_dfs = []
for i in range(1, 26):
    viol_df = dallas_cleaned.select(
        F.col("inspection_id_nk"),
        F.lit(i).alias("violation_order"),
        F.col(f"violation_description_{i}").alias("violation_description_raw"),
        F.col(f"violation_points_{i}").alias("violation_points_raw"),
        F.col(f"violation_detail_{i}").alias("violation_detail"),
        F.col(f"violation_memo_{i}").alias("violation_memo"),
        F.col("source_city"),
        F.current_timestamp().alias("load_ts"),
    ).filter(
        # Only keep slots that actually have a violation
        (F.col(f"violation_description_{i}").isNotNull()) &
        (F.trim(F.col(f"violation_description_{i}")) != "")
    )
    violation_dfs.append(viol_df)

dallas_violations_raw = reduce(DataFrame.unionAll, violation_dfs)

# Parse violation_code from description: extract *NN prefix
# Format: "*01 Cooling -- within 2 hours, 135-70F"
dallas_violations = dallas_violations_raw.select(
    F.col("inspection_id_nk"),
    F.col("violation_order"),
    # Extract code: digits after the asterisk
    F.regexp_extract(F.col("violation_description_raw"), r"^\*(\d+)", 1).alias("violation_code"),
    # Extract description: everything after "*NN " (strip the code prefix)
    F.regexp_extract(F.col("violation_description_raw"), r"^\*\d+\s+(.*)", 1).alias("violation_description"),
    # Clean corrupted degree symbol
    F.regexp_replace(F.col("violation_detail"), "ø", "\u00b0").alias("violation_detail"),
    F.regexp_replace(F.col("violation_memo"), "ø", "\u00b0").alias("violation_memo"),
    # Cast points to int
    F.col("violation_points_raw").cast("int").alias("points_assigned"),
    # Critical flag: points = 3 for Dallas
    F.when(F.col("violation_points_raw").cast("int") == 3, "Y")
     .otherwise("N").alias("is_critical_flag"),
    F.col("source_city"),
    F.col("load_ts"),
)

print(f"Dallas violations (unpivoted): {dallas_violations.count()} rows")
print(f"From {dallas_violations.select('inspection_id_nk').distinct().count()} inspections")

# Check parsing quality
print("\nSample parsed violations:")
dallas_violations.select(
    "inspection_id_nk", "violation_order", "violation_code",
    "violation_description", "points_assigned", "is_critical_flag"
).show(5, truncate=80)

# Check for empty violation_codes
empty_codes = dallas_violations.filter(
    (F.col("violation_code").isNull()) | (F.col("violation_code") == "")
).count()
print(f"Empty violation codes: {empty_codes}")

# Points distribution
print("\nPoints distribution:")
dallas_violations.groupBy("points_assigned").count().orderBy("points_assigned").show()

# Violations per inspection
print("\nViolations per inspection:")
dallas_violations.groupBy("inspection_id_nk").count().select(
    F.min("count").alias("min"),
    F.max("count").alias("max"),
    F.round(F.avg("count"), 1).alias("avg"),
    F.expr("percentile_approx(count, 0.5)").alias("median"),
).show()

Dallas violations (unpivoted): 397106 rows
From 72073 inspections

Sample parsed violations:
+--------------------------------+---------------+--------------+--------------------------------------------------------------------------------+---------------+----------------+
|                inspection_id_nk|violation_order|violation_code|                                                           violation_description|points_assigned|is_critical_flag|
+--------------------------------+---------------+--------------+--------------------------------------------------------------------------------+---------------+----------------+
|d65ae59663aa5d09e7455ab2614ec0fd|              1|            01|                                             Cooling -- within 2 hours, 135-70øF|              3|               Y|
|947f044987c89552c35f9c216a619e67|              1|            39|In-use utensils, between-use storage. During pauses in food preparation or di...|              1|               N|
|c073f4

In [0]:
# Cell 11: Investigate empty violation codes
from pyspark.sql import functions as F

# What do the unparseable violation descriptions look like?
dallas_violations.filter(
    (F.col("violation_code").isNull()) | (F.col("violation_code") == "")
).select("violation_description", "points_assigned").groupBy(
    "violation_description", "points_assigned"
).count().orderBy(F.desc("count")).show(20, truncate=100)

# Also show the raw descriptions that failed parsing
from pyspark.sql import DataFrame
from functools import reduce

# Rebuild just the raw descriptions for failed rows
failed_codes = dallas_violations.filter(
    (F.col("violation_code").isNull()) | (F.col("violation_code") == "")
)
print(f"\nFailed code extractions: {failed_codes.count()}")

# Check: do these have the asterisk prefix at all?
dallas_violations_raw.filter(
    F.regexp_extract(F.col("violation_description_raw"), r"^\*(\d+)", 1) == ""
).select("violation_description_raw").distinct().show(20, truncate=120)

+---------------------+---------------+-----+
|violation_description|points_assigned|count|
+---------------------+---------------+-----+
|                     |              2| 1517|
|                     |              1|   18|
|                     |              3|   10|
|                     |              4|    5|
+---------------------+---------------+-----+


Failed code extractions: 1550
+-------------------------------------------------+
|                        violation_description_raw|
+-------------------------------------------------+
|* 21 RFSM notify 10 days & replacement in 45 days|
|                           Sanitize all Equipment|
|                 Cleaning floors;spills, drippage|
|    Outdoor bars dispense ice thru dispenser only|
|        Outdoor bars single service articles only|
|   Prevent contamination by cleaning & sanitizing|
|                 Hair restraints/exempt employees|
|        In use utensil stored in water at < 135'F|
|                           

In [0]:
# Cell 12: Rebuild Dallas violations with fixed parsing
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql import DataFrame

# Rebuild from dallas_violations_raw (already created in Cell 10)
dallas_violations = dallas_violations_raw.select(
    F.col("inspection_id_nk"),
    F.col("violation_order"),
    
    # Fixed code extraction: allow optional space after asterisk
    F.when(
        F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*(\d+)", 1) != "",
        F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*(\d+)", 1)
    ).otherwise(F.lit("UNK")).alias("violation_code"),
    
    # Fixed description extraction: handle space after asterisk + no-code rows
    F.when(
        F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*\d+\s+(.*)", 1) != "",
        F.regexp_replace(
            F.regexp_extract(F.col("violation_description_raw"), r"^\*\s*\d+\s+(.*)", 1),
            "ø", "\u00b0"
        )
    ).otherwise(
        F.regexp_replace(F.trim(F.col("violation_description_raw")), "ø", "\u00b0")
    ).alias("violation_description"),
    
    # Clean degree symbol in detail and memo
    F.regexp_replace(F.col("violation_detail"), "ø", "\u00b0").alias("violation_detail"),
    F.regexp_replace(F.col("violation_memo"), "ø", "\u00b0").alias("violation_memo"),
    
    F.col("violation_points_raw").cast("int").alias("points_assigned"),
    
    F.when(F.col("violation_points_raw").cast("int") == 3, "Y")
     .otherwise("N").alias("is_critical_flag"),
    
    F.col("source_city"),
    F.col("load_ts"),
)

print(f"Dallas violations: {dallas_violations.count()} rows")

# Verify: how many UNK codes now?
unk_count = dallas_violations.filter(F.col("violation_code") == "UNK").count()
empty_count = dallas_violations.filter(
    (F.col("violation_code").isNull()) | (F.col("violation_code") == "")
).count()
print(f"UNK violation codes: {unk_count}")
print(f"Empty violation codes: {empty_count}")

# Verify degree symbol fix in descriptions
degree_check = dallas_violations.filter(
    F.col("violation_description").contains("ø")
).count()
print(f"Remaining corrupted degree symbols in description: {degree_check}")

# Sample the UNK rows
print("\nSample UNK violations:")
dallas_violations.filter(F.col("violation_code") == "UNK").select(
    "violation_code", "violation_description", "points_assigned"
).show(5, truncate=100)

# Sample the * 21 fix
print("\nCode 21 violations (was failing before):")
dallas_violations.filter(F.col("violation_code") == "21").select(
    "violation_code", "violation_description", "points_assigned"
).show(3, truncate=100)

Dallas violations: 397106 rows
UNK violation codes: 38
Empty violation codes: 0
Remaining corrupted degree symbols in description: 0

Sample UNK violations:
+--------------+----------------------------------------------+---------------+
|violation_code|                         violation_description|points_assigned|
+--------------+----------------------------------------------+---------------+
|           UNK|                        Sanitize all Equipment|              1|
|           UNK|              Cleaning floors;spills, drippage|              1|
|           UNK| Outdoor bars dispense ice thru dispenser only|              3|
|           UNK|     Outdoor bars single service articles only|              1|
|           UNK|Prevent contamination by cleaning & sanitizing|              2|
+--------------+----------------------------------------------+---------------+
only showing top 5 rows

Code 21 violations (was failing before):
+--------------+---------------------+---------------+
|v

In [0]:
# Cell 13: Apply Rule 6 and Rule 7 for Dallas
from pyspark.sql import functions as F

# Count violations and critical violations per inspection
dallas_viol_stats = dallas_violations.groupBy("inspection_id_nk").agg(
    F.count("*").alias("total_violations"),
    F.sum(F.when(F.col("is_critical_flag") == "Y", 1).otherwise(0)).alias("critical_count"),
)

# Join stats to inspection-level
dallas_with_stats = dallas_cleaned.join(
    dallas_viol_stats,
    on="inspection_id_nk",
    how="inner"
)

print(f"Before rules 6 & 7: {dallas_with_stats.count()}")

# Rule 6: score >= 90 cannot have more than 3 violations
rule6_violations = dallas_with_stats.filter(
    (F.col("inspection_score") >= 90) & (F.col("total_violations") > 3)
).count()
print(f"Rule 6 violations (score >= 90 with > 3 violations): {rule6_violations}")

# Rule 7: PASS (score >= 90) with critical violations (points = 3)
rule7_violations = dallas_with_stats.filter(
    (F.col("inspection_score") >= 90) & (F.col("critical_count") > 0)
).count()
print(f"Rule 7 violations (score >= 90 with critical): {rule7_violations}")

# Overlap: rows that fail BOTH rules
both_rules = dallas_with_stats.filter(
    (F.col("inspection_score") >= 90) & (F.col("total_violations") > 3) &
    (F.col("critical_count") > 0)
).count()
print(f"Overlap (fail both): {both_rules}")

# Apply both rules (drop rows failing either)
dallas_inspections_silver = dallas_with_stats.filter(
    ~(
        ((F.col("inspection_score") >= 90) & (F.col("total_violations") > 3)) |
        ((F.col("inspection_score") >= 90) & (F.col("critical_count") > 0))
    )
).drop("critical_count")

after_count = dallas_inspections_silver.count()
total_dropped = dallas_with_stats.count() - after_count
print(f"\nAfter rules 6 & 7: {after_count}")
print(f"Total dropped: {total_dropped}")

# Score distribution after filtering
print("\nScore distribution after all rules:")
dallas_inspections_silver.select(
    F.min("inspection_score").alias("min"),
    F.max("inspection_score").alias("max"),
    F.round(F.avg("inspection_score"), 1).alias("avg"),
    F.expr("percentile_approx(inspection_score, 0.5)").alias("median"),
).show()

Before rules 6 & 7: 72159
Rule 6 violations (score >= 90 with > 3 violations): 18294
Rule 7 violations (score >= 90 with critical): 24698
Overlap (fail both): 12989

After rules 6 & 7: 42156
Total dropped: 30003

Score distribution after all rules:
+---+---+----+------+
|min|max| avg|median|
+---+---+----+------+
| 44| 99|87.9|    87|
+---+---+----+------+



In [0]:
# Cell 14: Finalize and write Dallas Silver tables
from pyspark.sql import functions as F

# Get surviving inspection IDs
surviving_ids = dallas_inspections_silver.select("inspection_id_nk").distinct()

# Filter violations to surviving inspections
dallas_violations_silver = dallas_violations.join(
    surviving_ids,
    on="inspection_id_nk",
    how="inner"
)

# Final column selection for inspection table (drop wide violation columns and raw_violation_count)
dallas_inspections_final = dallas_inspections_silver.select(
    "inspection_id_nk",
    "restaurant_nk",
    "restaurant_name",
    "street_address",
    "street_number",
    "street_name",
    "street_direction",
    "street_type",
    "street_unit",
    "zip_code",
    "inspection_date",
    "inspection_type",
    "inspection_score",
    "results",
    "violation_score",
    "pass_fail_flag",
    "total_violations",
    "latitude",
    "longitude",
    "source_city",
    "load_ts",
)

# Final column selection for violations table
dallas_violations_final = dallas_violations_silver.select(
    "inspection_id_nk",
    "violation_order",
    "violation_code",
    "violation_description",
    "violation_detail",
    "violation_memo",
    "points_assigned",
    "is_critical_flag",
    "source_city",
    "load_ts",
)

# Write inspection table
dallas_inspections_final.write.mode("overwrite").saveAsTable(
    "food_inspections.silver.dallas_inspections"
)

# Write violations table
dallas_violations_final.write.mode("overwrite").saveAsTable(
    "food_inspections.silver.dallas_violations"
)

# Verify
di = spark.table("food_inspections.silver.dallas_inspections")
dv = spark.table("food_inspections.silver.dallas_violations")
print(f"Dallas Silver inspections: {di.count()} rows, {len(di.columns)} cols")
print(f"Dallas Silver violations:  {dv.count()} rows, {len(dv.columns)} cols")

# Every inspection should have at least 1 violation
zero_viol = di.filter(F.col("total_violations") == 0).count()
print(f"Inspections with 0 violations: {zero_viol}")

# Every violation should map to a surviving inspection
orphan_viols = dv.join(di.select("inspection_id_nk"), on="inspection_id_nk", how="left_anti").count()
print(f"Orphan violations: {orphan_viols}")

print("\nDallas Silver inspections schema:")
di.printSchema()

Dallas Silver inspections: 42156 rows, 21 cols
Dallas Silver violations:  281367 rows, 10 cols
Inspections with 0 violations: 0
Orphan violations: 0

Dallas Silver inspections schema:
root
 |-- inspection_id_nk: string (nullable = true)
 |-- restaurant_nk: string (nullable = true)
 |-- restaurant_name: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- street_number: string (nullable = true)
 |-- street_name: string (nullable = true)
 |-- street_direction: string (nullable = true)
 |-- street_type: string (nullable = true)
 |-- street_unit: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- inspection_date: date (nullable = true)
 |-- inspection_type: string (nullable = true)
 |-- inspection_score: integer (nullable = true)
 |-- results: string (nullable = true)
 |-- violation_score: double (nullable = true)
 |-- pass_fail_flag: string (nullable = true)
 |-- total_violations: long (nullable = true)
 |-- latitude: double (nullable = true)
 |--

In [0]:
# Cell 15: Silver layer summary
from pyspark.sql import functions as F

ci = spark.table("food_inspections.silver.chicago_inspections")
cv = spark.table("food_inspections.silver.chicago_violations")
di = spark.table("food_inspections.silver.dallas_inspections")
dv = spark.table("food_inspections.silver.dallas_violations")

print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)

print(f"\n{'Table':<45} {'Rows':>10} {'Cols':>6}")
print("-" * 60)
print(f"{'silver.chicago_inspections':<45} {ci.count():>10,} {len(ci.columns):>6}")
print(f"{'silver.chicago_violations':<45} {cv.count():>10,} {len(cv.columns):>6}")
print(f"{'silver.dallas_inspections':<45} {di.count():>10,} {len(di.columns):>6}")
print(f"{'silver.dallas_violations':<45} {dv.count():>10,} {len(dv.columns):>6}")

print("\n" + "=" * 60)
print("ROW REDUCTION FROM BRONZE")
print("=" * 60)
chi_bronze = 308357
dal_bronze = 78984
chi_silver = ci.count()
dal_silver = di.count()
print(f"\nChicago: {chi_bronze:,} -> {chi_silver:,} ({(chi_bronze - chi_silver):,} dropped, {(chi_bronze - chi_silver)/chi_bronze*100:.1f}%)")
print(f"Dallas:  {dal_bronze:,} -> {dal_silver:,} ({(dal_bronze - dal_silver):,} dropped, {(dal_bronze - dal_silver)/dal_bronze*100:.1f}%)")

print("\n" + "=" * 60)
print("VALIDATION RULES APPLIED")
print("=" * 60)
print("""
Chicago drops:
  - Null name/date/type/zip/results + zero violations: 86,484
  - Rule 7 (pure PASS with critical violations):          427
  
Dallas drops:
  - Null name/date/type + invalid zip + neg score + zero violations: 6,825
  - Rule 6 (score >= 90 with > 3 violations):            18,294
  - Rule 7 (score >= 90 with critical violations):       24,698
  - Overlap between Rule 6 and 7:                       -12,989
  - Net Rule 6+7 drop:                                   30,003
""")

print("=" * 60)
print("INTEGRITY CHECKS")
print("=" * 60)
# Cross-check: every inspection has violations, every violation has an inspection
chi_orphan_v = cv.join(ci.select("inspection_id_nk"), on="inspection_id_nk", how="left_anti").count()
chi_zero_v = ci.filter(F.col("total_violations") == 0).count()
dal_orphan_v = dv.join(di.select("inspection_id_nk"), on="inspection_id_nk", how="left_anti").count()
dal_zero_v = di.filter(F.col("total_violations") == 0).count()
print(f"\nChicago orphan violations: {chi_orphan_v}")
print(f"Chicago inspections with 0 violations: {chi_zero_v}")
print(f"Dallas orphan violations: {dal_orphan_v}")
print(f"Dallas inspections with 0 violations: {dal_zero_v}")
print(f"\nAll checks passed: {chi_orphan_v == 0 and chi_zero_v == 0 and dal_orphan_v == 0 and dal_zero_v == 0}")

SILVER LAYER SUMMARY

Table                                               Rows   Cols
------------------------------------------------------------
silver.chicago_inspections                       221,446     22
silver.chicago_violations                      1,000,213      8
silver.dallas_inspections                         42,156     21
silver.dallas_violations                         281,367     10

ROW REDUCTION FROM BRONZE

Chicago: 308,357 -> 221,446 (86,911 dropped, 28.2%)
Dallas:  78,984 -> 42,156 (36,828 dropped, 46.6%)

VALIDATION RULES APPLIED

Chicago drops:
  - Null name/date/type/zip/results + zero violations: 86,484
  - Rule 7 (pure PASS with critical violations):          427
  
Dallas drops:
  - Null name/date/type + invalid zip + neg score + zero violations: 6,825
  - Rule 6 (score >= 90 with > 3 violations):            18,294
  - Rule 7 (score >= 90 with critical violations):       24,698
  - Overlap between Rule 6 and 7:                       -12,989
  - Net Rule 6+7 